In [ ]:
%load_ext autoreload
%autoreload 2

from concrete_music_16 import ConcreteMusic16, Sounds
from soundmining_library.piece import Piece

piece = Piece()
await piece.start(should_send_to_score=False)

helpers = ConcreteMusic16(piece)

In [ ]:
%%html
<style>
.cell-output-ipywidget-background {
   background-color: transparent !important;
}
.jp-OutputArea-output {
   background-color: transparent;
}  
</style>

In [ ]:
from soundmining_library.ui.ui_controls import UiControls

piece.reset()

all_sound_data = helpers._all_sound_data
s2_sound = all_sound_data[Sounds.S2]

for sound_data in all_sound_data.values():
    sound_data.add_sound_to_synth_player(piece.synth_player)

ui_controls = (
    UiControls(piece)
    .header_label("Sound Partials")
    .partial_canvas(all_sound_data[Sounds.S2])
    .header_label("Spectrums")
    .spectrum_canvas(helpers._overtone_spectrum1, f"S2 overtone spectrum 1 {helpers._fact1:.4f}", use_log_scale=False)
    .spectrum_canvas(helpers._undertone_spectrum1, f"S2 undertone spectrum 1 {helpers._fact1:.4f}", use_log_scale=False)
    .spectrum_canvas(helpers._overtone_spectrum2, f"S2 overtone spectrum 2 {helpers._fact2:.4f}", use_log_scale=False)
    .spectrum_canvas(helpers._undertone_spectrum3, f"S2 undertone spectrum 3 {helpers._fact3:.4f}", use_log_scale=False)
    .header_label("Sounds")
    .sound_grid()
    .stop_button()
    .render()
)


piece.synth_player.start()

In [ ]:
from soundmining_library.supercollider_receiver import ExtendedNoteHandler, PatchArguments


In [ ]:
from soundmining_library.generative import random_range
from soundmining_library.supercollider_client import SupercolliderClient


class MyNoteHandler(ExtendedNoteHandler):
    def __init__(self, client: SupercolliderClient):
        self._ui_controls = ui_controls
        super().__init__(client)

    def handle_note(self, patch_arguments: PatchArguments):
        play_rate = random_range(3, 5)
        match patch_arguments.octave:
            case 2:
                helpers.play_high_pad1(
                    patch_arguments.start,
                    patch_arguments.note,
                    patch_arguments.amp,
                    helpers._s2_sound_data,
                    helpers._overtone_spectrum1,
                    rate_note=2,
                    play_rate=play_rate,
                )
            case 3:
                helpers.play_low_pad1(
                    patch_arguments.start,
                    patch_arguments.note,
                    patch_arguments.amp,
                    helpers._s2_sound_data,
                    helpers._undertone_spectrum1,
                    rate_note=10,
                    play_rate=play_rate,
                )
            case 4:
                helpers.play_high_pad1(
                    patch_arguments.start,
                    patch_arguments.note,
                    patch_arguments.amp,
                    helpers._s2_sound_data,
                    helpers._overtone_spectrum2,
                    rate_note=2,
                    play_rate=play_rate,
                )
            case 5:
                helpers.play_low_pad1(
                    patch_arguments.start,
                    patch_arguments.note,
                    patch_arguments.amp,
                    helpers._s2_sound_data,
                    helpers._undertone_spectrum3,
                    rate_note=10,
                    play_rate=play_rate,
                )


my_handler = MyNoteHandler(piece.supercollider_client)
piece.receiver.set_note_handler(my_handler)

In [ ]:
class MyMelodyHandler(ExtendedNoteHandler):
    def __init__(self, client: SupercolliderClient):
        self._ui_controls = ui_controls
        super().__init__(client)

    def handle_note(self, patch_arguments: PatchArguments) -> None:
        match patch_arguments.note:
            case 0:
                helpers.play_high_pad_melody1(patch_arguments.start)
            case 1:
                helpers.play_low_pad_melody1(patch_arguments.start)
            case 2:
                helpers.play_low_pad_melody2(patch_arguments.start)
            case _:
                pass


my_handler = MyMelodyHandler(piece.supercollider_client)
piece.receiver.set_note_handler(my_handler)

In [ ]:
await piece.stop()